# Tries

## 1. Trie Fundamentals

A **trie** (prefix tree) is a tree that stores a set of strings by decomposing them character-by-character along edges.

- Each node represents a character position; each edge is labeled with a single character
- A path from root to any node spells a **prefix** of one or more stored strings
- A node marked `isWord=True` is a **terminal node** -- the path from root to it spells a complete stored word
- Words sharing a prefix share the same path from root, so no prefix is stored more than once

**TrieNode attributes:**
- `children`: hash map (char -> TrieNode)
- `isWord`: boolean, True if a complete word ends here
- **Alternative storage:** fixed array of size 26 (lowercase-only), or a `word` string attribute at terminal nodes (stores the full word for direct collection in board search problems)

### Terminology

| Term | Definition |
|------|-----------|
| Prefix tree | Tree where each root-to-node path represents a prefix of stored strings |
| TrieNode | Node with a `children` map (char -> TrieNode) and an end-of-word marker |
| `isWord` | Boolean at a node: True if a complete word ends here |
| Prefix sharing | Words with a common prefix reuse the same nodes along that path |
| Terminal node | A node where `isWord` is True |

### Structure

```text
T.insert("bat"), T.insert("byte"), T.insert("bytes"):

        root
       /
      b
     / \
    a    y
    |    |
    t*   t
         |
         e*
         |
         s*

* = isWord
```

"byte" and "bytes" share `b -> y -> t -> e`. "bat" branches at `b`.

### Properties

| Property | Detail |
|---|---|
| **Key idea** | Tree storing strings by shared prefixes, $O(k)$ operations where k = word length |
| **Use-case** | Autocomplete, prefix matching, word search, spell checking, IP routing |
| **Time** | $O(k)$ for `T.insert()`, `T.search()`, `T.startsWith()` |
| **Space** | $O(N \cdot k)$ worst case for N words of avg length k, reduced by prefix sharing |

### Operations

| Operation | Description | Time |
|-----------|-------------|------|
| `T.insert(word)` | Add word, creating nodes as needed, mark terminal | $O(k)$ |
| `T.search(word)` | Return True if word exists as a complete word | $O(k)$ |
| `T.startsWith(prefix)` | Return True if any stored word starts with prefix | $O(k)$ |
| `delete(T, word)` | Traverse to terminal node, unmark `isWord` | $O(k)$ |
| `autocomplete(T, prefix)` | Return all words starting with prefix | $O(p + m)$ |

$k$ = word/prefix length, $p$ = prefix length, $m$ = total chars in matching words

## 2. Trie

`T.insert()`, `T.search()`, and `T.startsWith()` share the same traversal: start at `root`, iterate characters, advance through `children` one level per character. They diverge only at the end:

- `T.insert(word)`: create missing child nodes, set `isWord=True` at the last node
- `T.search(word)`: return `node.isWord` at the last node
- `T.startsWith(prefix)`: return True if the traversal completes without a missing child

In [1]:
class TrieNode:
    def __init__(self):
        self.children = {}
        self.isWord = False


class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.isWord = True

    def search(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                return False
            node = node.children[ch]
        return node.isWord

    def startsWith(self, prefix):
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return False
            node = node.children[ch]
        return True


T = Trie()
for w in ["top", "bye", "byte"]:
    T.insert(w)
print(T.search("top"), T.search("to"), T.search("byte"))
print(T.startsWith("by"), T.startsWith("ba"))

True False True
True False


#### Steps

`T = Trie()`, insert "top", "bye", "byte".

| Step | Operation | Action | Result |
|------|-----------|--------|--------|
| 1 | `T.insert("top")` | create t -> o -> p, mark p.isWord | path added |
| 2 | `T.insert("bye")` | create b -> y -> e, mark e.isWord | new branch from root |
| 3 | `T.insert("byte")` | reuse b -> y, create t -> e, mark e.isWord | branches at y: e* and t -> e* |
| 4 | `T.search("top")` | traverse t -> o -> p, p.isWord=True | True |
| 5 | `T.search("to")` | traverse t -> o, o.isWord=False | False ("to" is a prefix, not a word) |
| 6 | `T.search("byte")` | traverse b -> y -> t -> e, e.isWord=True | True |
| 7 | `T.startsWith("by")` | traverse b -> y, node exists | True |
| 8 | `T.startsWith("ba")` | at b, 'a' not in children | False |

```text
final trie:

        root
       /    \
      t      b
      |      |
      o      y
      |     / \
      p*   e*  t
                |
                e*
```

## 3. Extended Operations

Three operations that build on the core traversal:

- **`delete(T, word)`:** traverse to the terminal node, set `node.isWord=False`. Does not prune empty branches -- nodes remain in case other words share the path.
- **`collectWords(node, prefix)`:** DFS from `node`, accumulate characters along the path, append the built string whenever `cur.isWord` is True.
- **`autocomplete(T, prefix)`:** navigate to the node at the end of the prefix (same as `T.startsWith()` traversal), then `collectWords()` from that node.

In [2]:
def delete(trie, word):
    node = trie.root
    for ch in word:
        if ch not in node.children:
            return
        node = node.children[ch]
    node.isWord = False


def collectWords(node, prefix):
    res = []

    def dfs(cur, path):
        if cur.isWord:
            res.append(path)
        for ch, child in cur.children.items():
            dfs(child, path + ch)

    dfs(node, prefix)
    return res


def autocomplete(trie, prefix):
    node = trie.root
    for ch in prefix:
        if ch not in node.children:
            return []
        node = node.children[ch]
    return collectWords(node, prefix)


T = Trie()
for w in ["app", "apple", "ape", "bat"]:
    T.insert(w)
print(autocomplete(T, "ap"))
delete(T, "app")
print(T.search("app"), T.search("apple"))
print(autocomplete(T, "ap"))

['app', 'apple', 'ape']
False True
['apple', 'ape']


#### Steps

Trie contains "app", "apple", "ape", "bat".

**autocomplete("ap"):**

| Step | Node | Path | Action |
|------|------|------|--------|
| 1 | root -> a -> p | "ap" | prefix found, start DFS from p |
| 2 | p -> p (isWord=True) | "app" | append "app" |
| 3 | p -> p -> l -> e (isWord=True) | "apple" | append "apple" |
| 4 | p -> e (isWord=True) | "ape" | append "ape" |

Result: `["app", "apple", "ape"]`

**delete("app"):** traverse a -> p -> p, set p.isWord=False. The node still exists (has child l for "apple"), just unmarked.

**autocomplete("ap") after delete:** same DFS, but p.isWord is now False. Result: `["apple", "ape"]`

## 4. Wildcard Search

A modified `T.search()` where `'.'` matches any single character. When `'.'` is encountered, recursively explore all children. Uses a helper `dfs(node, i)` to track position without restarting from root.

- Without wildcards: $O(k)$, same as `T.search()`
- With all wildcards (e.g. `"..."` on a trie with 26-letter branching): $O(26^k)$ worst case

In [3]:
def searchWithWildcard(trie, word):

    def dfs(node, i):
        if i == len(word):
            return node.isWord
        ch = word[i]
        if ch == ".":
            for child in node.children.values():
                if dfs(child, i + 1):
                    return True
            return False
        if ch not in node.children:
            return False
        return dfs(node.children[ch], i + 1)

    return dfs(trie.root, 0)


T = Trie()
for w in ["bad", "dad", "mad", "bat"]:
    T.insert(w)
print(searchWithWildcard(T, "b.d"))
print(searchWithWildcard(T, "..d"))
print(searchWithWildcard(T, "b.."))
print(searchWithWildcard(T, "b.z"))

True
True
True
False


#### Steps

Trie contains "bad", "dad", "mad", "bat".

**search "..d":**

| Step | i | ch | Node | Action |
|------|---|-----|------|--------|
| 1 | 0 | '.' | root | wildcard: try all children (b, d, m) |
| 2 | 1 | '.' | b | wildcard: try children (a) |
| 3 | 2 | 'd' | a | 'd' in children -> move to d |
| 4 | 3 | -- | d | i == len(word), d.isWord=True -> True |

First branch (b -> a -> d) succeeds, return True immediately.

**search "b.z":**

| Step | i | ch | Node | Action |
|------|---|-----|------|--------|
| 1 | 0 | 'b' | root | 'b' in children -> move to b |
| 2 | 1 | '.' | b | wildcard: try child (a) |
| 3 | 2 | 'z' | a | 'z' not in children -> False |
| 4 | -- | -- | -- | no more children at b -> False |

## 5. Bitwise Trie

A bitwise trie stores integers bit-by-bit. Each `BitNode` has `children[0]` and `children[1]`. Numbers are inserted from MSB (bit 31) down to LSB (bit 0), forming a binary tree of fixed depth 32.

**Maximum XOR:** `BT.getMax(num)` traverses the trie greedily picking the opposite bit at each level to maximize the XOR. Each query is $O(32) = O(1)$. Building the trie from $n$ numbers is $O(32n)$.

In [4]:
class BitNode:
    def __init__(self):
        self.children = [None, None]


class BitTrie:
    def __init__(self):
        self.root = BitNode()

    def insert(self, num):
        node = self.root
        for i in range(31, -1, -1):
            bit = (num >> i) & 1
            if not node.children[bit]:
                node.children[bit] = BitNode()
            node = node.children[bit]

    def getMax(self, num):
        node = self.root
        xor = 0
        for i in range(31, -1, -1):
            bit = (num >> i) & 1
            want = 1 - bit
            if node.children[want]:
                xor |= 1 << i
                node = node.children[want]
            else:
                node = node.children[bit]
        return xor


def maxXOR(nums):
    bt = BitTrie()
    for num in nums:
        bt.insert(num)
    res = 0
    for num in nums:
        res = max(res, bt.getMax(num))
    return res


print(maxXOR([3, 10, 5, 25, 2, 8]))

28


#### Steps

`maxXOR([5, 25])` using 5-bit representation for readability.

5 = `00101`, 25 = `11001`

**getMax(5):** for each bit of 5, pick the opposite bit if available in the trie.

| Bit pos | bit of 5 | want | available | pick | XOR bit |
|---------|----------|------|-----------|------|---------|
| 4 | 0 | 1 | yes (25's path) | 1 | 1 |
| 3 | 0 | 1 | yes | 1 | 1 |
| 2 | 1 | 0 | yes | 0 | 1 |
| 1 | 0 | 1 | no, pick 0 | 0 | 0 |
| 0 | 1 | 0 | no, pick 1 | 1 | 0 |

XOR = `11100` = 28. Confirms: 5 XOR 25 = 28.

## 6. Patterns

| Pattern | Description | Problem keywords |
|---------|-------------|-----------------|
| **Word search on board** | Build trie from word list, DFS/backtrack from each cell | "word search II", "find all words on board" |
| **Counting trie** | Track pass-through count and end count per node | "count words with prefix", "implement trie II" |
| **Distinct substrings** | Insert all suffixes of a string, count total trie nodes | "count distinct substrings" |
| **Longest common prefix** | Insert all strings, walk trie while node has exactly one child | "longest common prefix" |

All pattern implementations below reuse the `TrieNode` class from section 2.

### Word Search on Board

Given an m x n board of characters and a list of words, find all words that can be formed by tracing adjacent cells (up, down, left, right). Each cell is used at most once per word.

- Build a trie from the word list, storing the full word string at terminal nodes (`node.word = "..."`) for direct collection
- DFS/backtrack from each cell matching a root child
- Mark visited cells with `'#'`, restore on backtrack
- Set `node.word = None` after recording to avoid duplicate results

In [5]:
def findWords(board, words):
    root = TrieNode()
    for w in words:
        node = root
        for ch in w:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.word = w

    m, n = len(board), len(board[0])
    res = []

    def dfs(r, c, node):
        ch = board[r][c]
        if ch not in node.children:
            return
        child = node.children[ch]
        if getattr(child, "word", None):
            res.append(child.word)
            child.word = None
        board[r][c] = "#"
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and board[nr][nc] != "#":
                dfs(nr, nc, child)
        board[r][c] = ch

    for r in range(m):
        for c in range(n):
            dfs(r, c, root)
    return res


board = [
    ["o", "a", "a", "n"],
    ["e", "t", "a", "e"],
    ["i", "h", "k", "r"],
    ["i", "f", "l", "v"],
]
print(findWords(board, ["oath", "pea", "eat", "rain"]))

['oath', 'eat']


#### Steps

Trie built from ["oath", "eat"]. Board search finds paths matching trie branches.

| Step | Cell | Path in trie | Action |
|------|------|-------------|--------|
| 1 | (0,0) 'o' | root -> o | 'o' in `root.children`, start DFS |
| 2 | (0,1) 'a' | o -> a | 'a' in `o.children`, continue |
| 3 | (1,1) 't' | o -> a -> t | 't' in `a.children`, continue |
| 4 | (2,1) 'h' | o -> a -> t -> h | `h.word="oath"`, record it, set `word=None` |
| 5 | (1,0) 'e' | root -> e | new DFS from cell (1,0) |
| 6 | (0,1) 'a' -> (0,2) 'a' -> ... | e -> a -> t | `t.word="eat"`, record it |

Result: `["oath", "eat"]`

**Complexity:** $O(N \cdot L + m \cdot n \cdot 3^L)$
- $N \cdot L$ to build the trie from $N$ words of max length $L$
- $m \cdot n \cdot 3^L$ for DFS: start from each of $m \cdot n$ cells, branch into 3 directions per step (excludes the cell just came from), up to $L$ levels deep

### Counting Trie

Augments each `CountNode` with two counters:
- `count`: number of inserted words whose path passes through this node
- `endsWith`: number of inserted words that end at this node

Supports `CT.countWordsStartingWith(prefix)`, `CT.countWordsEqualTo(word)`, and `CT.erase(word)` which decrements counts along the path.

In [6]:
class CountNode:
    def __init__(self):
        self.children = {}
        self.count = 0
        self.endsWith = 0


class CountTrie:
    def __init__(self):
        self.root = CountNode()

    def insert(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = CountNode()
            node = node.children[ch]
            node.count += 1
        node.endsWith += 1

    def countWordsEqualTo(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                return 0
            node = node.children[ch]
        return node.endsWith

    def countWordsStartingWith(self, prefix):
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return 0
            node = node.children[ch]
        return node.count

    def erase(self, word):
        node = self.root
        for ch in word:
            node = node.children[ch]
            node.count -= 1
        node.endsWith -= 1


ct = CountTrie()
ct.insert("apple")
ct.insert("apple")
ct.insert("app")
print(ct.countWordsEqualTo("apple"))
print(ct.countWordsStartingWith("app"))
ct.erase("apple")
print(ct.countWordsEqualTo("apple"))
print(ct.countWordsStartingWith("app"))

2
3
1
2


#### Steps

Insert "apple", "apple", "app".

| Step | Operation | Node path (count, endsWith at terminal) |
|------|-----------|----------------------------------------|
| 1 | insert "apple" | a(1) -> p(1) -> p(1) -> l(1) -> e(1, endsWith=1) |
| 2 | insert "apple" | a(2) -> p(2) -> p(2) -> l(2) -> e(2, endsWith=2) |
| 3 | insert "app" | a(3) -> p(3) -> p(3, endsWith=1) |
| 4 | countWordsEqualTo("apple") | traverse to e, endsWith=2 -> 2 |
| 5 | countWordsStartingWith("app") | traverse to second p, count=3 -> 3 |
| 6 | erase("apple") | decrement: a(2) p(2) p(2) l(1) e(1, endsWith=1) |
| 7 | countWordsEqualTo("apple") | e.endsWith=1 -> 1 |
| 8 | countWordsStartingWith("app") | second p.count=2 -> 2 |

### Other Patterns

**Distinct substrings:**

Insert all suffixes of a string into a trie. Each new node created during suffix insertion corresponds to a unique substring.

- For "aba": insert suffixes "aba", "ba", "a"
- New nodes: a, ab, aba, b, ba -> 5 distinct substrings
- Time: $O(k^2)$ to insert all suffixes of a string of length $k$

**Longest common prefix:**

Insert all strings into a trie. Walk from `root`, advancing one level at a time while:
- `node` has exactly one child
- `node.isWord` is False (no word ends mid-prefix)

The characters collected along this walk form the longest common prefix.

## 7. Reference

### Operation Complexity

| Operation | Time | Notes |
|-----------|------|-------|
| Insert | $O(k)$ | k = length of word, create at most k new nodes |
| Search | $O(k)$ | Traverse k levels, each step is $O(1)$ hash map lookup |
| StartsWith | $O(k)$ | Same traversal as search, check node existence instead of isWord |
| Delete | $O(k)$ | Traverse to node, unmark isWord |
| Autocomplete | $O(p + m)$ | p = prefix length, m = total characters in all matching words |
| Wildcard search | $O(k)$ to $O(26^k)$ | $O(k)$ without wildcards, $O(26^k)$ worst case with all '.' |
| Bitwise max XOR | $O(32) = O(1)$ | Fixed 32-bit depth per query |

### Trie vs Other Structures

| Structure | Prefix search | Exact search | Insert | Space | Ordered |
|-----------|--------------|-------------|--------|-------|---------|
| Trie | $O(k)$ | $O(k)$ | $O(k)$ | $O(N \cdot k)$ | By prefix |
| Hash table | $O(N)$ scan all keys | $O(1)$ avg | $O(1)$ avg | $O(N \cdot k)$ | No |
| Sorted array | $O(\log N + k)$ | $O(\log N + k)$ | $O(N)$ shift | $O(N \cdot k)$ | Yes |
| BST (balanced) | $O(k \cdot \log N)$ | $O(k \cdot \log N)$ | $O(k \cdot \log N)$ | $O(N \cdot k)$ | Yes |

N = number of stored words, k = word length.

### Pattern Quick Reference

| Pattern | Trie variant | Time | Use-case | Problem keywords |
|---------|-------------|------|----------|-----------------|
| Prefix search / autocomplete | Standard trie + DFS | $O(p + m)$ | Find all words matching a prefix | "autocomplete", "search suggestions", "prefix" |
| Wildcard search | Standard trie + recursive DFS | $O(26^k)$ worst | Match patterns with '.' wildcards | "add and search word", "design word dictionary" |
| Word search on board | Trie + backtracking | $O(m \cdot n \cdot 3^L)$ | Find dictionary words on a character grid | "word search II", "boggle" |
| Maximum XOR | Bitwise trie (binary) | $O(n \cdot 32)$ | Find pair with maximum XOR in an array | "maximum XOR of two numbers", "maximum xor with element" |
| Counting queries | Counting trie | $O(k)$ per query | Count words equal to or starting with a string | "count prefix", "implement trie II" |
| Distinct substrings | Standard trie + all suffixes | $O(k^2)$ build | Count unique substrings of a string | "distinct substrings", "number of substrings" |
| Longest common prefix | Standard trie | $O(N \cdot k)$ build + $O(k)$ walk | Find longest shared prefix of all strings | "longest common prefix" |

### Cross-References

- Hash table alternative for exact-match lookups: see `hash_tables.ipynb`
- DFS/backtracking patterns used in word search: see `trees.ipynb`